## Estatísticas via Understat API
Understat expõe os dados via endpoint AJAX (`POST /main/getPlayersStats/`).
Sem scraping de HTML, sem async, sem dependências exóticas — apenas `requests`.

In [3]:
import requests
import time
import unicodedata
import re
import pandas as pd
from rapidfuzz import process, fuzz
from pathlib import Path

base_path = Path('/Users/luccagazotto/Documents/Personal/ML Projects/Soccer player/Soccer_player_market_value')

HEADERS = {
    'User-Agent': 'Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) AppleWebKit/537.36 (KHTML, like Gecko) Chrome/120.0.0.0 Safari/537.36',
    'Referer': 'https://understat.com/',
    'X-Requested-With': 'XMLHttpRequest',
    'Content-Type': 'application/x-www-form-urlencoded',
}

In [4]:
# ── 1. Coleta de dados ───────────────────────────────────────────────────────
# Understat usa o ano de início da temporada (2017 = 2017-18)
# Igual à convenção do Transfermarkt (saison_id=2017)

def get_understat_season(year: int) -> pd.DataFrame:
    resp = requests.post(
        'https://understat.com/main/getPlayersStats/',
        headers=HEADERS,
        data={'league': 'La_liga', 'season': str(year)},
        timeout=30,
    )
    resp.raise_for_status()
    players = resp.json().get('players', [])
    df = pd.DataFrame(players)
    df['season_year'] = year
    return df


years = list(range(2017, 2025))  # temporadas 2017-18 até 2024-25
seasons = []

for y in years:
    print(f'Coletando {y}-{y+1}...', end=' ')
    df_s = get_understat_season(y)
    seasons.append(df_s)
    print(f'{len(df_s)} jogadores')
    time.sleep(1.5)  # respeita rate limit

understat_raw = pd.concat(seasons, ignore_index=True)
print(f'\nTotal bruto: {understat_raw.shape}')
understat_raw.head()

Coletando 2017-2018... 557 jogadores
Coletando 2018-2019... 531 jogadores
Coletando 2019-2020... 556 jogadores
Coletando 2020-2021... 570 jogadores
Coletando 2021-2022... 603 jogadores
Coletando 2022-2023... 584 jogadores
Coletando 2023-2024... 598 jogadores
Coletando 2024-2025... 590 jogadores

Total bruto: (4589, 19)


,id,player_name,games,time,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position,team_title,npg,npxG,xGChain,xGBuildup,season_year
0,2097,Lionel Messi,36,2995,34,28.946291390806437,12,15.100405801087618,196,87,3,0,F M S,Barcelona,32,25.973176393657923,48.18063086271286,21.63439449854195,2017
1,2371,Cristiano Ronaldo,27,2304,26,26.999309986829758,5,5.524839024990797,178,40,1,0,F,Real Madrid,23,24.026196867227554,31.62383684515953,9.471848987042904,2017
2,2098,Luis Suárez,33,2902,25,25.36797896027565,12,11.144682643935084,121,56,9,0,F S,Barcelona,24,24.624704957008362,38.56682434864342,8.494499390944839,2017
3,2290,Iago Aspas,34,2939,22,17.546523582190275,5,8.858793137595057,94,67,9,0,F M S,Celta Vigo,20,15.317330490797758,31.42611799389124,11.639251209795475,2017
4,1709,Cristhian Stuani,33,2728,21,18.785554103553295,0,0.957874481100589,73,12,8,0,F S,Girona,16,15.069159049540758,11.786210827063769,1.4602142125368118,2017


In [5]:
# ── 2. Limpeza e tipagem ─────────────────────────────────────────────────────
understat_df = understat_raw.rename(columns={
    'season_year': 'year',
    'player_name': 'player_name_us',
    'team_title':  'team_us',
    'position':    'position_us',
    'time':        'minutes',
}).copy()

numeric_cols = ['games', 'minutes', 'goals', 'assists', 'shots',
                'key_passes', 'yellow_cards', 'red_cards', 'npg',
                'xG', 'xA', 'npxG', 'xGChain', 'xGBuildup']
for col in [c for c in numeric_cols if c in understat_df.columns]:
    understat_df[col] = pd.to_numeric(understat_df[col], errors='coerce')

understat_df.to_parquet(base_path / 'understat_stats.parquet', index=False)
print(f'Salvo: understat_stats.parquet  shape={understat_df.shape}')
understat_df.head()

Salvo: understat_stats.parquet  shape=(4589, 19)


,id,player_name_us,games,minutes,goals,xG,assists,xA,shots,key_passes,yellow_cards,red_cards,position_us,team_us,npg,npxG,xGChain,xGBuildup,year
0,2097,Lionel Messi,36,2995,34,28.946291,12,15.100406,196,87,3,0,F M S,Barcelona,32,25.973176,48.180631,21.634394,2017
1,2371,Cristiano Ronaldo,27,2304,26,26.999310,5,5.524839,178,40,1,0,F,Real Madrid,23,24.026197,31.623837,9.471849,2017
2,2098,Luis Suárez,33,2902,25,25.367979,12,11.144683,121,56,9,0,F S,Barcelona,24,24.624705,38.566824,8.494499,2017
3,2290,Iago Aspas,34,2939,22,17.546524,5,8.858793,94,67,9,0,F M S,Celta Vigo,20,15.317330,31.426118,11.639251,2017
4,1709,Cristhian Stuani,33,2728,21,18.785554,0,0.957874,73,12,8,0,F S,Girona,16,15.069159,11.786211,1.460214,2017


In [6]:
# ── 3. Normalização de nomes para fuzzy join ─────────────────────────────────
# Remove acentos, lowercase, pontuação — maximiza matching entre TM e Understat

def normalize_name(name: str) -> str:
    if not isinstance(name, str):
        return ''
    name = unicodedata.normalize('NFKD', name)
    name = ''.join(c for c in name if not unicodedata.combining(c))
    name = name.lower().strip()
    name = re.sub(r'[^a-z ]', '', name)
    name = re.sub(r'\s+', ' ', name)
    return name


understat_df['name_norm'] = understat_df['player_name_us'].apply(normalize_name)

tm_df = pd.read_parquet(base_path / 'players_panel.parquet')
tm_df['name_norm'] = tm_df['player_name'].apply(normalize_name)
tm_df['year'] = tm_df['year'].astype(int)

print(f'Transfermarkt: {tm_df.shape}')
print(f'Understat:     {understat_df.shape}')

Transfermarkt: (6386, 12)
Understat:     (4589, 20)


In [7]:
# ── 4. Fuzzy join por temporada ──────────────────────────────────────────────
# token_sort_ratio: tolera ordem dos tokens
# ex: 'marc andre ter stegen' vs 'ter stegen marc andre' → score alto
#
# Jogadores sem match (~25%) são jogadores emprestados fora da La Liga
# ou reserves com poucos minutos — ficam com NaN nas stats, comportamento correto.

STAT_COLS = [c for c in understat_df.columns
             if c not in ['year', 'name_norm', 'id']]


def fuzzy_join_season(tm_s: pd.DataFrame, us_s: pd.DataFrame,
                      threshold: int = 80) -> pd.DataFrame:
    us_names = us_s['name_norm'].tolist()
    rows = []
    for _, row in tm_s.iterrows():
        result = process.extractOne(
            row['name_norm'],
            us_names,
            scorer=fuzz.token_sort_ratio,
            score_cutoff=threshold,
        )
        merged = row.to_dict()
        if result:
            _, score, idx = result
            merged.update(us_s.iloc[idx][STAT_COLS].to_dict())
            merged['match_score'] = score
        else:
            merged['match_score'] = None
        rows.append(merged)
    return pd.DataFrame(rows)


all_seasons = []
for year in sorted(tm_df['year'].unique()):
    tm_s = tm_df[tm_df['year'] == year].copy()
    us_s = understat_df[understat_df['year'] == year].copy()

    if us_s.empty:
        print(f'{year}: sem dados Understat')
        all_seasons.append(tm_s)
        continue

    merged = fuzzy_join_season(tm_s, us_s, threshold=80)
    matched = merged['match_score'].notna().sum()
    pct = 100 * matched / len(merged)
    print(f'{year}: {matched}/{len(merged)} matched ({pct:.0f}%)')
    all_seasons.append(merged)

panel_final = pd.concat(all_seasons, ignore_index=True)
print(f'\nPanel final: {panel_final.shape}')

2017: 509/674 matched (76%)
2018: 491/678 matched (72%)
2019: 516/704 matched (73%)
2020: 539/734 matched (73%)
2021: 585/783 matched (75%)
2022: 574/754 matched (76%)
2023: 585/774 matched (76%)
2024: 578/785 matched (74%)
2025: sem dados Understat

Panel final: (6386, 30)


In [ ]:
# ── 5. Remover matches problemáticos ─────────────────────────────────────────
# Matches com score < 85 são potencialmente falsos positivos.
# Melhor ter NaN nas stats do que stats do jogador errado.

mask_bad = panel_final['match_score'].notna() & (panel_final['match_score'] < 85)
print(f'Matches com score < 85 (anulados): {mask_bad.sum()}')

stat_cols = ['player_name_us', 'team_us', 'position_us', 'games', 'minutes',
             'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes',
             'yellow_cards', 'red_cards', 'npg', 'npxG', 'xGChain', 'xGBuildup']
stat_cols_existing = [c for c in stat_cols if c in panel_final.columns]

panel_final.loc[mask_bad, stat_cols_existing] = None
panel_final.loc[mask_bad, 'match_score'] = None

print(f'Jogadores com stats: {panel_final["match_score"].notna().sum()}')
print(f'Jogadores sem stats: {panel_final["match_score"].isna().sum()}')

In [9]:
# ── 6. Salvar painel final ───────────────────────────────────────────────────
panel_final.to_parquet(base_path / 'panel_final.parquet', index=False)
print('Salvo: panel_final.parquet')
print(f'Shape: {panel_final.shape}')
print(f'Colunas: {list(panel_final.columns)}')
panel_final.head()

Salvo: panel_final.parquet
Shape: (6386, 30)
Colunas: ['year', 'club', 'player_id', 'player_name', 'link_html', 'age', 'height', 'nationality', 'foot', 'position', 'market_value', 'name_norm', 'player_name_us', 'games', 'minutes', 'goals', 'xG', 'assists', 'xA', 'shots', 'key_passes', 'yellow_cards', 'red_cards', 'position_us', 'team_us', 'npg', 'npxG', 'xGChain', 'xGBuildup', 'match_score']


,year,club,player_id,player_name,link_html,age,height,nationality,foot,position,...,key_passes,yellow_cards,red_cards,position_us,team_us,npg,npxG,xGChain,xGBuildup,match_score
0,2017,FC Barcelona,74857,Marc-André ter Stegen,https://www.transfermarkt.us/marc-andre-ter-st...,25.0,1.87,Germany,right,Goalkeeper,...,0.0,0.0,0.0,GK,Barcelona,0.0,0.000000,11.089831,11.089831,100.0
1,2017,FC Barcelona,146227,Jasper Cillessen,https://www.transfermarkt.us/jasper-cillessen/...,28.0,1.87,Netherlands,right,Goalkeeper,...,0.0,0.0,0.0,GK,Barcelona,0.0,0.000000,0.092318,0.092318,100.0
2,2017,FC Barcelona,142033,Adrián Ortolá,https://www.transfermarkt.us/adrian-ortola/pro...,24.0,1.87,Spain,left,Goalkeeper,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,2017,FC Barcelona,126540,Samuel Umtiti,https://www.transfermarkt.us/samuel-umtiti/pro...,24.0,1.82,France,left,Defender - Centre-Back,...,1.0,7.0,0.0,D S,Barcelona,1.0,0.944605,13.100010,13.004152,100.0
4,2017,FC Barcelona,18944,Gerard Piqué,https://www.transfermarkt.us/gerard-pique/prof...,30.0,1.94,Spain,right,Defender - Centre-Back,...,6.0,9.0,0.0,D S,Barcelona,2.0,2.634030,18.464774,16.717330,100.0
